# QeFEM paper experiments

This notebook is the single local entry point for the paper results. It uses the validated implementation in `qefem_prof`; it does **not** reimplement the algorithm. The experiment harness supplies Feng's tuned parameterization, schedules, optimizer settings, seed protocol, coherent trajectory capture, and resumable reporting.

Running all cells produces `results/{k2000,wishart,chook-tile,gset,gset-big}`. Every completed case is written immediately. Imported laptop runs are retained as `imported_local_*` evidence, while an incomplete case is rerun automatically. Raw candidate populations are intentionally omitted from the paper workspace.

> The full suite is compute intensive. `SMOKE=True` checks the complete pipeline cheaply; `SMOKE=False` is the paper run. CPU is the exact portable default.

In [1]:
from pathlib import Path
import sys

PAPER_ROOT = Path('/Users/ronit/Desktop/QeFEM/Paper')
sys.path.insert(0, str(PAPER_ROOT))

from paper_experiments import (
    RESULTS_ROOT, GSET_ALL, GSET_BIG, K2000_REPLICAS,
    import_existing_local_results, initialize_runtime, run_k2000,
    run_planted_family, run_gset, write_root_report,
)

# Canonical paper settings. Change only DEVICE if required by this laptop.
DEVICE = 'cpu'          # exact portable default; 'mps' is faster but a new platform run
DTYPE = 'float32'
THREADS = 10
STRICT_REVISION = True  # pins qefem_prof to the validated source revision
SMOKE = False           # True: four-step pipeline check; False: paper protocol
OVERWRITE = False       # resumable; True intentionally replaces canonical outputs
INCLUDE_LQA = True

RUN_K2000 = True
RUN_WISHART = True
RUN_CHOOK = True
RUN_GSET = True
RUN_GSET_BIG = True

print('G-set G1-G54 instances:', len(GSET_ALL))
print('Supplied G55+ instances:', GSET_BIG)
print('K2000 replica counts:', K2000_REPLICAS)

G-set G1-G54 instances: 54
Supplied G55+ instances: ['G55', 'G56', 'G57', 'G58', 'G59', 'G60', 'G61', 'G62', 'G63', 'G64', 'G65', 'G66', 'G67', 'G70', 'G72', 'G77', 'G81']
K2000 replica counts: [128, 512, 1024, 2048, 4096, 8192]


## 1. Seed the dojo with completed local evidence

This is idempotent. It preserves notebook-completed canonical results and refreshes only the `imported_local_*` reference files. The 22 exact replay-checked K2000/Wishart/Chook trajectories are available immediately; older G-set Adam runs are labeled as historical baselines rather than Feng-tuned paper results.

In [2]:
import_summary = import_existing_local_results()
import_summary

{'complete_cases': 22, 'gset_endpoints': 71, 'gset_trajectories': 29}

## 2. Pin and initialize the runtime

The revision check prevents an unnoticed code change from being mixed into the paper tables. Deterministic settings are installed before experiment tensors are created.

In [3]:
runtime = initialize_runtime(
    device=DEVICE, dtype_name=DTYPE, threads=THREADS,
    strict_revision=STRICT_REVISION,
)
write_root_report(runtime)
runtime

Runtime(device=device(type='cpu'), dtype=torch.float32, dtype_name='float32', threads=10, revision='b14f73c53305b55242b418ef2013fba0ac3cce54')

## 3. K2000 population study

Runs nested prefixes of the same randomized population at 128, 512, 1,024, 2,048, 4,096, and 8,192 replicas. This makes the population comparison controlled rather than seed-confounded. Five endpoints are recorded per population, plus LQA. The best local endpoint gets a coherent final-winner trajectory containing only discrete cut, target gap, and entropy.

In [4]:
k2000 = (run_k2000(runtime, overwrite=OVERWRITE, smoke=SMOKE, include_lqa=INCLUDE_LQA)
         if RUN_K2000 else None)
k2000

k2000/replicas-00128: complete
k2000/replicas-00512: complete
k2000/replicas-01024: complete
k2000/replicas-02048: complete
k2000/replicas-04096: complete
k2000/replicas-08192: complete


,method,trial,seed,energy,cut,gap,relative_error,ground_hit,target_hit,best_replica,runtime_sec,replicas
0,qefem,0,3371835386,-67550.0,33255.0,82.0,NaN,NaN,False,18,14.319148,128.0
1,qefem,1,3700479728,-67540.0,33250.0,87.0,NaN,NaN,False,93,14.420219,128.0
2,qefem,2,1934420243,-67540.0,33250.0,87.0,NaN,NaN,False,20,11.686070,128.0
3,qefem,3,716463040,-67540.0,33250.0,87.0,NaN,NaN,False,81,11.159248,128.0
4,qefem,4,424211264,-67550.0,33255.0,82.0,NaN,NaN,False,11,11.471273,128.0
5,qefem,0,3371835386,-67550.0,33255.0,82.0,NaN,NaN,False,18,43.589648,512.0
6,qefem,1,3700479728,-67546.0,33253.0,84.0,NaN,NaN,False,304,54.381915,512.0
7,qefem,2,1934420243,-67550.0,33255.0,82.0,NaN,NaN,False,316,51.071404,512.0
8,qefem,3,716463040,-67550.0,33255.0,82.0,NaN,NaN,False,465,53.042824,512.0
9,qefem,4,424211264,-67550.0,33255.0,82.0,NaN,NaN,False,11,50.623138,512.0


## 4. Wishart planted-solution suite

Runs every supplied 500-spin instance from $\alpha=0.1$ to $1.0$, with five QeFEM and five LQA endpoints per instance. Each `alpha-*` folder contains complete settings, seeds, endpoints, a report, and best-run trajectories.

In [5]:
wishart = (run_planted_family('wishart', runtime, overwrite=OVERWRITE,
                               smoke=SMOKE, include_lqa=INCLUDE_LQA)
           if RUN_WISHART else None)
wishart

wishart/alpha-0.1: complete
wishart/alpha-0.2: complete
wishart/alpha-0.3: complete
wishart/alpha-0.4: complete
wishart/alpha-0.5: complete
wishart/alpha-0.6: complete
wishart/alpha-0.7: complete
wishart/alpha-0.8: complete
wishart/alpha-0.9: complete
wishart/alpha-1.0: complete


,method,trial,seed,energy,cut,gap,relative_error,ground_hit,target_hit,best_replica,runtime_sec
0,qefem,0,3289218920,-24.832100,None,0.187080,0.007477,False,None,3883,71.044119
1,lqa,0,3289218920,-24.444052,None,0.575128,0.022987,False,None,0,0.103497
2,qefem,1,3345798870,-24.838459,None,0.180720,0.007223,False,None,5835,73.963734
3,lqa,1,3345798870,-24.415468,None,0.603712,0.024130,False,None,0,0.105181
4,qefem,2,92671970,-24.844564,None,0.174616,0.006979,False,None,819,71.403654
...,...,...,...,...,...,...,...,...,...,...,...
95,lqa,2,3468624555,-218.903964,None,31.281346,0.125033,False,None,0,0.108046
96,qefem,3,65934964,-250.185310,None,0.000000,0.000000,True,None,42,78.858894
97,lqa,3,65934964,-219.096708,None,31.088602,0.124262,False,None,0,0.105986
98,qefem,4,2781859735,-250.185310,None,0.000000,0.000000,True,None,2,81.554486


## 5. Chook tile-planted suite

Runs every supplied 1,024-spin instance across $p(C_3)=0.0,0.1,\ldots,1.0$. The two frozen seed batches give ten QeFEM and ten LQA endpoints per instance.

In [6]:
chook = (run_planted_family('chook', runtime, overwrite=OVERWRITE,
                             smoke=SMOKE, include_lqa=INCLUDE_LQA)
         if RUN_CHOOK else None)
chook

chook/p-c3-0.0: complete
chook/p-c3-0.1: complete
chook/p-c3-0.2: complete
chook/p-c3-0.3: complete
chook/p-c3-0.4: complete
chook/p-c3-0.5: complete
chook/p-c3-0.6: complete
chook/p-c3-0.7: complete
chook/p-c3-0.8: complete
chook/p-c3-0.9: complete
chook/p-c3-1.0: complete


,method,trial,seed,energy,cut,gap,relative_error,ground_hit,target_hit,best_replica,runtime_sec
0,qefem,0,3494023143,-2032.0,None,16.0,0.007812,False,None,3442,158.200826
1,lqa,0,3494023143,-1986.0,None,62.0,0.030273,False,None,0,0.120631
2,qefem,1,1077125151,-2032.0,None,16.0,0.007812,False,None,5477,151.870934
3,lqa,1,1077125151,-2008.0,None,40.0,0.019531,False,None,0,0.117361
4,qefem,2,3170439463,-2032.0,None,16.0,0.007812,False,None,1978,151.302219
...,...,...,...,...,...,...,...,...,...,...,...
215,lqa,7,2856244789,-1536.0,None,0.0,0.000000,True,None,0,0.121920
216,qefem,8,980884606,-1536.0,None,0.0,0.000000,True,None,1,179.874844
217,lqa,8,980884606,-1536.0,None,0.0,0.000000,True,None,0,0.120510
218,qefem,9,4201941292,-1536.0,None,0.0,0.000000,True,None,1,179.991640


## 6. Complete G-set G1--G54 suite

Runs every instance from G1 through G54. For each graph, the workflow uses its frozen discrete/manual evaluation plan when one exists; otherwise it uses the frozen smooth evaluation plan. The topology labels follow the supplied classical-FEM benchmark table. Each graph is saved immediately, so the cell can be resumed safely with `OVERWRITE=False`.

In [8]:
gset = (run_gset(runtime, big=False, overwrite=OVERWRITE, smoke=SMOKE)
        if RUN_GSET else None)
gset

gset/G10: complete
gset/G24: complete
gset/G25: complete
gset/G46: complete
gset/G47: complete
gset/G12: complete
gset/G13: complete
gset/G48: complete
gset/G49: complete
gset/G50: complete
gset/G18: complete
gset/G19: complete
gset/G20: complete
gset/G21: complete
gset/G54: complete


,graph,topology,trial,seed_base,seed,replicas,steps,target,best_cut,gap,winner,target_hits,runtime_sec
0,G10,random,0,93000,4227048712,4096,2000,2000.0,2000.0,0.0,129,17,941.827579
1,G10,random,1,93001,3366346643,4096,2000,2000.0,2000.0,0.0,63,14,908.775339
2,G10,random,2,93002,1232737513,4096,2000,2000.0,2000.0,0.0,88,23,985.987627
3,G10,random,3,93003,981061398,4096,2000,2000.0,2000.0,0.0,1167,20,952.312265
4,G10,random,4,93004,932889664,4096,2000,2000.0,2000.0,0.0,237,14,972.205666
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,G21,planar,0,96200,4151780881,4096,4000,931.0,931.0,0.0,1008,1,407.592985
58,G21,planar,1,96201,527818180,4096,4000,931.0,931.0,0.0,3599,2,406.607678
59,G21,planar,2,96202,1082231800,4096,4000,931.0,931.0,0.0,2235,2,405.819468
60,G54,planar,0,96200,3339604113,2048,4000,3852.0,3847.0,5.0,1326,0,272.439212


## 7. Large G-set suite (all supplied instances above G54)

Runs G55--G67 plus G70, G72, G77, and G81: every `G>54` file supplied in this checkout. These graphs are large; the cell saves each graph atomically and can be stopped and resumed.

In [ ]:
gset_big = (run_gset(runtime, big=True, overwrite=OVERWRITE, smoke=SMOKE)
            if RUN_GSET_BIG else None)
gset_big

gset-big/G55: complete


## 8. Final audit

The audit lists manifests and missing required artifacts without modifying experiment data. A paper run is complete when every enabled case has a canonical `manifest.json`, `configuration.json`, `endpoints.csv`, `trajectory.csv`, and `report.md` (LQA trajectories use `trajectory_lqa.csv`).

In [ ]:
import pandas as pd

write_root_report(runtime)
rows = []
for manifest in sorted(RESULTS_ROOT.glob('*/*/manifest.json')):
    folder = manifest.parent
    trajectory_name = 'trajectory_lqa.csv' if folder.name == 'lqa' else 'trajectory.csv'
    required = ['configuration.json', 'endpoints.csv', 'report.md', trajectory_name]
    has_trajectory = (folder / trajectory_name).exists()
    rows.append({
        'case': str(folder.relative_to(RESULTS_ROOT)),
        'status': __import__('json').loads(manifest.read_text()).get('status'),
        'trajectory': has_trajectory,
        'missing': ', '.join(name for name in required if not (folder / name).exists()),
    })
audit = pd.DataFrame(rows)
audit.to_csv(RESULTS_ROOT / 'artifact_audit.csv', index=False)
display(audit)
print('Results:', RESULTS_ROOT)